# 03 — Evaluation

Score the lite QLoRA adapter on the **1,000 test cars**. These rows were not used in training or in the validation loss.

Checkpoint **500** had the best validation loss, so this notebook loads `checkpoint-500` when that folder is present. Each predictor sees only the prompt, which ends at `Price is EGP`. The real price is used afterwards for MAE and MAPE.

Four predictors, in order:

1. Mean price of the 8,000 lite training ads
2. Median price of those same ads
3. `Qwen/Qwen2.5-3B` in 4-bit, no adapter
4. That model plus the lite LoRA adapter

Run this on a Colab **T4**. A close guess is within 15% of the asking price. Within 30% is the middle band.

## 1. Setup

On Colab, install the train extra, then restart the runtime if this is the first install and run from the next cell.

The adapter is not in git. If this is a new Colab session, upload `qwen2.5-3b-lite.zip` to `/content/` before the adapter cell. The test split is rebuilt from the Hub when `data/test.parquet` is missing. Seed 42 keeps the same 1,000 cars.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
REPO_URL = "https://github.com/MohamedAlaa2180/egyptian-car-pricer.git"


def find_root() -> Path | None:
    here = Path.cwd().resolve()
    candidates = [here, here.parent, Path("/content/egyptian-car-pricer"), Path("/content")]
    for candidate in candidates:
        if (candidate / "pricer" / "prep.py").exists():
            return candidate
    return None


if IN_COLAB and find_root() is None:
    dest = Path("/content/egyptian-car-pricer")
    subprocess.check_call(["git", "clone", REPO_URL, str(dest)])
    os.chdir(dest)

ROOT = find_root()
if ROOT is None:
    raise SystemExit("Repo root not found. Open this notebook from the project, or clone it on Colab.")
os.chdir(ROOT)
print("Repo:", ROOT)

if IN_COLAB:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", ".[train]"])
    print("Installed .[train]. Restart the runtime if this was the first install, then rerun from the next cell.")
else:
    print("Local runtime. The model section needs a CUDA GPU.")

## 2. GPU check

The two constant baselines do not need a GPU. The Qwen sections do. This cell stops on a CPU before any model download.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "pricer" / "prep.py").exists() and (ROOT.parent / "pricer" / "prep.py").exists():
    ROOT = ROOT.parent.resolve()
    os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ["TOKENIZERS_PARALLELISM"] = "false"

try:
    import torch
except ImportError as exc:
    raise SystemExit(
        "torch is not installed. On Colab, run the setup cell and restart the runtime."
    ) from exc

from dotenv import load_dotenv
from huggingface_hub import login

if not torch.cuda.is_available():
    raise SystemExit("No CUDA GPU. Use a Colab T4: Runtime → Change runtime type → T4 GPU.")

props = torch.cuda.get_device_properties(0)
total_gb = getattr(props, "total_gb", None) or props.total_memory / 1024**3
print(f"GPU: {torch.cuda.get_device_name(0)}  ({total_gb:.1f} GB)")

load_dotenv(ROOT / ".env", override=True)
token = os.getenv("HF_TOKEN")
if not token and "COLAB_RELEASE_TAG" in os.environ:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = None
if token:
    login(token, add_to_git_credential=False)
    print("Logged in to Hugging Face")
else:
    print("No HF_TOKEN. The public base model can still download.")

## 3. Load the test cars

The model is shown `test_prompt()`, which stops at `Price is EGP`. The completion stays in the row only so we can score the guess.

In [ ]:
import pandas as pd

from pricer.items import PREFIX
from pricer.prep import (
    LITE_TRAIN_SIZE,
    SOURCE_DATASET,
    apply_hard_filters,
    apply_price_outliers,
    drop_exact_duplicates,
    lite_train,
    load_source,
    parse_frame,
    save_splits,
    split_frame,
    to_items,
)

DATA_DIR = ROOT / "data"


def rebuild_splits(data_dir):
    raw = load_source()
    parsed, n_fail = parse_frame(raw)
    kept, dropped_hard = apply_hard_filters(parsed)
    kept, n_dup = drop_exact_duplicates(kept)
    kept, dropped_out = apply_price_outliers(kept)
    train, val, test = split_frame(kept)
    lite = lite_train(train, n=LITE_TRAIN_SIZE)
    report = {
        "source_dataset": SOURCE_DATASET,
        "source_rows": int(len(raw)),
        "parse_failed": int(n_fail),
        "exact_duplicates": int(n_dup),
        "price_outliers": int(len(dropped_out)),
        "kept": int(len(kept)),
        "splits": {
            "train": int(len(train)),
            "train_lite": int(len(lite)),
            "val": int(len(val)),
            "test": int(len(test)),
        },
    }
    save_splits(train, val, test, report, lite=lite, data_dir=data_dir)
    print(f"Rebuilt splits. Test cars: {len(test):,}")


if not (DATA_DIR / "test.parquet").exists() or not (DATA_DIR / "train_lite.parquet").exists():
    print("Local parquet missing. Rebuilding from the Hub.")
    rebuild_splits(DATA_DIR)

train_lite_df = pd.read_parquet(DATA_DIR / "train_lite.parquet")
test_df = pd.read_parquet(DATA_DIR / "test.parquet")
test_items = to_items(test_df)
print(f"lite train {len(train_lite_df):,}   test {len(test_items):,}")
print("\n--- shown to the model ---")
print(test_items[0].test_prompt())
print("--- hidden price ---")
print(test_items[0].completion)
if PREFIX not in test_items[0].test_prompt() or test_items[0].completion in test_items[0].test_prompt():
    raise SystemExit("The test prompt is leaking the price or missing the EGP prefix.")

## 4. Constant baselines

The mean and the median guess the same EGP amount for every car. They use the 8,000 lite prices, because that is the set the adapter was trained on. Beating the median on MAE is the bar for "this fine-tune learned a price."

In [ ]:
from pricer.evaluator import Tester

MEAN_PRICE = float(train_lite_df["price"].mean())
MEDIAN_PRICE = float(train_lite_df["price"].median())
print(f"lite mean   EGP {MEAN_PRICE:,.0f}")
print(f"lite median EGP {MEDIAN_PRICE:,.0f}")


def train_lite_mean(_item):
    return MEAN_PRICE


def train_lite_median(_item):
    return MEDIAN_PRICE


def score(name, predictor, items):
    guesses = [Tester.post_process(predictor(item)) for item in items]
    truths = [float(item.price) for item in items]
    errors = [abs(g - t) for g, t in zip(guesses, truths)]
    mae = sum(errors) / len(errors)
    mape = sum((e / t) if t else 1.0 for e, t in zip(errors, truths)) / len(errors) * 100
    within_15 = sum((e / t) < 0.15 for e, t in zip(errors, truths) if t) / len(errors) * 100
    within_30 = sum((e / t) < 0.30 for e, t in zip(errors, truths) if t) / len(errors) * 100
    print(f"{name}: MAE EGP {mae:,.0f}   MAPE {mape:.1f}%   within 15% {within_15:.1f}%   within 30% {within_30:.1f}%")
    return {"name": name, "mae": mae, "mape": mape, "within_15": within_15, "within_30": within_30, "guesses": guesses}


results = []
results.append(score("train lite mean", train_lite_mean, test_items))
results.append(score("train lite median", train_lite_median, test_items))
Tester(train_lite_mean, test_items, size=len(test_items), workers=1).run()
Tester(train_lite_median, test_items, size=len(test_items), workers=1).run()

## 5. Base Qwen, then the adapter

Both models are greedy: the same prompt always gives the same digits. New tokens are capped at 12, which is enough for an EGP amount. The prompt itself is not decoded, so a year inside the spec sheet cannot be mistaken for the prediction.

The adapter path prefers `checkpoint-500`, because that step had the lowest validation loss. If the folder only has the final `adapter_model.safetensors` from the save cell, that file is the same weights.

In [ ]:
import shutil
from pathlib import Path

from peft import PeftModel
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-3B"
MAX_PROMPT_LENGTH = 256
MAX_NEW_TOKENS = 12
GEN_BATCH = 8

zip_path = Path("/content/qwen2.5-3b-lite.zip")
unzip_dir = Path("/content/adapters/qwen2.5-3b-lite")
if zip_path.exists() and not unzip_dir.exists():
    unzip_dir.mkdir(parents=True, exist_ok=True)
    shutil.unpack_archive(zip_path, unzip_dir)
    print("Unzipped", zip_path, "to", unzip_dir)


def find_adapter() -> Path:
    candidates = [
        Path("/content/adapters/qwen2.5-3b-lite"),
        ROOT / "adapters" / "qwen2.5-3b-lite",
        Path("/content/qwen2.5-3b-lite"),
    ]
    for start in candidates:
        if not start.exists():
            continue
        preferred = start / "checkpoint-500"
        if (preferred / "adapter_config.json").exists():
            return preferred
        if (start / "adapter_config.json").exists():
            return start
        for config in start.rglob("adapter_config.json"):
            if "checkpoint-500" in config.parts:
                return config.parent
        found = next(start.rglob("adapter_config.json"), None)
        if found is not None:
            return found.parent
    raise SystemExit(
        "Adapter not found. Upload qwen2.5-3b-lite.zip to /content/ "
        "or keep /content/adapters/qwen2.5-3b-lite from the training session."
    )


def generate_prices(model, tokenizer, items):
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    prompts = [item.test_prompt() for item in items]
    guesses = []
    model.eval()
    for start in tqdm(range(0, len(prompts), GEN_BATCH), desc="generate"):
        batch = prompts[start : start + GEN_BATCH]
        encoded = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_PROMPT_LENGTH,
        ).to(model.device)
        with torch.inference_mode():
            output = model.generate(
                **encoded,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )
        prompt_width = encoded["input_ids"].shape[1]
        for row in output:
            text = tokenizer.decode(row[prompt_width:], skip_special_tokens=True)
            guesses.append(text.strip())
    return guesses


def cached_predictor(name, guesses):
    by_item = {id(item): guess for item, guess in zip(test_items, guesses)}

    def predict(item):
        return by_item[id(item)]

    predict.__name__ = name
    return predict


USE_BF16 = torch.cuda.is_bf16_supported()
COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    dtype=COMPUTE_DTYPE,
)
base_guesses = generate_prices(base, tokenizer, test_items)
print("base sample:", base_guesses[0], " actual:", test_items[0].completion)
results.append(score("base qwen", cached_predictor("base_qwen", base_guesses), test_items))
Tester(cached_predictor("base_qwen", base_guesses), test_items, size=len(test_items), workers=1).run()

adapter_path = find_adapter()
print("Loading adapter:", adapter_path)
adapted = PeftModel.from_pretrained(base, str(adapter_path))
adapter_guesses = generate_prices(adapted, tokenizer, test_items)
print("adapter sample:", adapter_guesses[0], " actual:", test_items[0].completion)
results.append(score("qlora checkpoint 500", cached_predictor("qlora_checkpoint_500", adapter_guesses), test_items))
Tester(cached_predictor("qlora_checkpoint_500", adapter_guesses), test_items, size=len(test_items), workers=1).run()

## 6. Compare

MAE is the average absolute error in EGP. MAPE is that error as a percent of the asking price. The adapter is useful when its MAE is below the lite median. Within 15% and within 30% are the green and orange bands from `pricer/evaluator.py`.

In [ ]:
summary = pd.DataFrame(results)[["name", "mae", "mape", "within_15", "within_30"]]
summary = summary.sort_values("mae")
display(summary)
best = summary.iloc[0]
print(f"Lowest MAE: {best['name']}  EGP {best['mae']:,.0f}")